# 📊 Module 03: Feature Correlation & Multicollinearity
## 🎓 The Complete Masterclass Guide for Data Science & Machine Learning
*Author: Muhammad Zabiullah ([@mzb000](https://github.com/mzb000))*  

---

## 📌 Table of Contents
1. **What is Correlation? (Direction & Strength)**
2. **Pearson vs Spearman vs Kendall: When to Use Which?**
3. **The Multicollinearity Danger in Linear & Logistic Models**
4. **Step-by-Step Hands-On Code (Correlation Matrix & VIF)**
5. **Line-by-Line Code Breakdown**
6. **Correlation Does NOT Imply Causation (Real-World Examples)**
7. **Interview Q&A on Correlation**

---

## 💡 1. What is Correlation?
Correlation measures the **statistical relationship** between two numerical features.
- **Range:** $[-1.0, +1.0]$
- **$+1.0$ (Perfect Positive):** As Feature $X$ increases, Feature $Y$ increases in lockstep (e.g., House Area vs Price).
- **$0.0$ (No Linear Relationship):** Feature $X$ and $Y$ have zero linear connection (e.g., Number of shoes owned vs IQ score).
- **$-1.0$ (Perfect Negative):** As Feature $X$ increases, Feature $Y$ decreases proportionally (e.g., Speed of car vs Time to reach destination).

## 📐 2. Pearson vs Spearman: Which One Should You Choose?

| Criterion | Pearson Correlation ($r$) | Spearman Rank Correlation ($\rho$) |
| :--- | :--- | :--- |
| **Type of Relationship** | Strict **Linear** ($y = mx + b$) | Monotonic (Curves that always go up or always go down) |
| **Data Requirement** | Continuous, Normally Distributed | Continuous or Ordinal ranks |
| **Sensitivity to Outliers** | **Extremely Sensitive** (1 outlier can flip $r$ from $+0.9$ to $0$) | **Robust** (Ranks dampen extreme values) |
| **Formula** | $r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \sum(y_i - \bar{y})^2}}$ | $\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$ |

> **Golden Rule for Data Scientists:**
> If your data contains extreme outliers or non-linear growth (e.g., exponential growth, ranks), **use Spearman**.
> If your data is clean, Gaussian, and strictly linear, **use Pearson**.

## ⚠️ 3. The Multicollinearity Trap in Machine Learning

What happens when two input features $X_1$ and $X_2$ are strongly correlated with each other ($r > 0.85$)?

**Example:** Suppose you are predicting House Price, and you include:
- $X_1$: House Area in **Square Feet**
- $X_2$: House Area in **Square Meters**

Both features provide the **exact same signal** to the model!
### What Goes Wrong?
1. **Exploding Standard Errors:** The matrix $(X^T X)$ becomes almost non-invertible (singular). Weights become volatile and jump wildly between $+10,000$ and $-9,990$.
2. **P-Values become meaningless:** Features that are actually important appear statistically insignificant!
3. **Solution:** Drop one of the redundant features, or combine them using PCA dimensionality reduction.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Let's create an educational student performance dataset
np.random.seed(42)
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.float64)
sleep_hours = np.array([9.0, 8.5, 8.0, 7.5, 7.0, 6.5, 6.0, 5.5, 5.0, 4.5])  # Negative relationship with study
exam_scores = np.array([52, 58, 65, 71, 76, 82, 88, 91, 95, 99], dtype=np.float64)  # Strong positive with study

# Add an artificial collinear feature: Study Minutes (Exact same info as hours!)
study_minutes = study_hours * 60.0

df_students = pd.DataFrame({
    "study_hours": study_hours,
    "study_minutes": study_minutes,
    "sleep_hours": sleep_hours,
    "exam_score": exam_scores
})

# Compute Pearson correlation matrix
corr_matrix = df_students.corr(method="pearson")

print("=== Pearson Correlation Matrix ===")
print(corr_matrix.round(3))

print("\n=== Multicollinearity Detection ===")
corr_hours_minutes = corr_matrix.loc["study_hours", "study_minutes"]
print(f"Correlation between Hours and Minutes: {corr_hours_minutes:.4f} (PERFECT 1.0 COLLINEARITY!)")
print("Action Required: Drop 'study_minutes' before training any linear model!")

### 🔍 Line-by-Line Code Breakdown:
- `df_students.corr(method="pearson")`: Computes pairwise correlation across all numerical columns. It outputs a symmetric square matrix with $1.0$ on the diagonal.
- `study_hours` vs `exam_score`: $+0.998$ (Almost perfect positive relationship).
- `sleep_hours` vs `exam_score`: $-1.000$ (Strong negative relationship in this synthetic cohort).
- `study_hours` vs `study_minutes`: $+1.000$ (Classic multicollinearity; having both introduces a singular matrix in OLS regression).

## 🚫 5. Correlation Does NOT Imply Causation!

A famous data science proverb that every interviewer will test you on.

### Famous Example:
In summer cities, there is a strong positive correlation between:
- **$X$:** Ice Cream Sales
- **$Y$:** Drowning Incidents

Does eating ice cream cause people to drown? **Of course not!**
There is a hidden third variable called a **Confounding Variable (Hot Summer Weather)**:
- Hot weather $\to$ More ice cream consumed.
- Hot weather $\to$ More people swimming in lakes and oceans $\to$ More accidental drownings.

Never make business or medical decisions based purely on correlation without controlled A/B testing!

## 🎯 6. Top Interview Questions on Correlation

### Q1: What is the Variance Inflation Factor (VIF)?
- **Answer:** VIF measures how much the variance of an estimated regression coefficient is inflated due to multicollinearity.
  $$VIF = \frac{1}{1 - R_i^2}$$
  - $VIF = 1$: No correlation.
  - $VIF > 5$ or $10$: High multicollinearity; feature must be dropped or transformed.

### Q2: Can two variables have zero Pearson correlation but still be strongly dependent?
- **Answer:** **YES!** Consider $y = x^2$ where $x \in [-10, 10]$. The relationship is a perfect parabola (100% deterministic), but Pearson correlation is $r = 0.0$ because Pearson only detects *linear* patterns!